# Topic Modeling with LDA

Did chunking in hopes of getting better topic models

## Setup and Imports

In [242]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation as LDA, NMF 

from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA, TruncatedSVD as SVD

In [243]:
sns.set_theme(style="white")
colors = "YlGnBu"

In [244]:
model_type = 'lda' # or 'nmf'
data_home = "../input"


In [245]:
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

In [246]:
OHCO = ['doc_title', 'chunk_id','token_id']
CHUNKS = OHCO[:2]
STORIES = OHCO[:1]

BAG = CHUNKS

In [247]:
LIB = pd.read_csv('data/pg2591-LIB.csv',)
LIB.set_index('doc_title', inplace=True)
LIB.head()

,volume
doc_title,
THE GOLDEN BIRD,1
HANS IN LUCK,1
JORINDA AND JORINDEL,1
THE TRAVELLING MUSICIANS,1
OLD SULTAN,1


In [248]:
TOKENS = pd.read_csv('data/chunked_tokens.csv').set_index(OHCO).dropna()
TOKENS

pos_tuple  pos token_str term_str pos_group
doc_title chunk_id token_id                                                   
ASHPUTTEL 0        0           ('the', 'DT')   DT       the      the        DT
                   1          ('wife', 'NN')   NN      wife     wife        NN
                   2            ('of', 'IN')   IN        of       of        IN
                   3             ('a', 'DT')   DT         a        a        DT
                   4          ('rich', 'JJ')   JJ      rich     rich        JJ
...                                      ...  ...       ...      ...       ...
TOM THUMB 19       66           ('s', 'VBZ')  VBZ         s        s        VB
                   67           ('no', 'DT')   DT        no       no        DT
                   68        ('place', 'NN')   NN     place    place        NN
                   69         ('like', 'IN')   IN      like     like        IN
                   70         ('home', 'NN')   NN      home     home        NN

[115213 rows x 5 columns]

In [249]:
DOCS = TOKENS[TOKENS.pos.str.match(r'^NNS?$')]\
    .groupby(BAG).term_str\
    .apply(lambda x: ' '.join(map(str,x)))\
    .to_frame()\
    .rename(columns={'term_str':'doc_str'})

DOCS

doc_str
doc_title chunk_id                                                   
ASHPUTTEL 0         wife man end drew daughter bedside girl i watc...
          1         fair face foul heart sorry time girl goodforno...
          2         hearth ashes course dirty ashputtel father wif...
          3         daughter mother s grave tears tree times day b...
          4         hair shoes sashes king s feast ball mother not...
...                                                               ...
TOM THUMB 15        wolf chat friend i treat s wolf house father s...
          16        content way tom shout noise wolf everybody hou...
          17        wolf woodman axe wife scythe do woodman i head...
          18        ah father world i way home air father i mouseh...
          19        clothes ones journey home father mother peace ...

[784 rows x 1 columns]

## Create Vector Space

In [250]:
from sklearn.feature_extraction import text

my_stop_words = list(text.ENGLISH_STOP_WORDS.union(['yes']))
my_stop_words[:10]
# my_stop_words.append('said') # add more words to stop words because they appeared it most topics and ruined the topics
# my_stop_words.append('came')
# my_stop_words.append('went')

['nothing',
 'put',
 'himself',
 'otherwise',
 'these',
 'in',
 'between',
 'except',
 'will',
 'how']

In [251]:
count_engine = CountVectorizer(max_df=.75, min_df=5, stop_words=my_stop_words) # Got some advice from clause to lower min ax max df because corpus ins amll
count_model = count_engine.fit_transform(DOCS.doc_str)
TERMS = count_engine.get_feature_names_out()
VOCAB = pd.DataFrame(index=TERMS)
VOCAB.index.name = 'term_str'
DTM_chunk = pd.DataFrame(count_model.toarray(), index=DOCS.index, columns=TERMS)
DTM_chunk

account  advice  ah  air  alas  ale  anger  animals  \
doc_title chunk_id                                                        
ASHPUTTEL 0               0       0   0    0     0    0      0        0   
          1               0       0   0    0     0    0      0        0   
          2               0       0   0    0     0    0      0        0   
          3               0       0   0    0     0    0      0        0   
          4               0       0   0    0     0    0      0        0   
...                     ...     ...  ..  ...   ...  ...    ...      ...   
TOM THUMB 15              0       0   0    0     0    0      0        0   
          16              0       0   0    0     0    0      0        0   
          17              0       0   1    0     0    0      0        0   
          18              0       0   1    1     0    0      0        0   
          19              0       0   0    0     0    0      0        0   

                    answer  apple  ...  woods  word  words  work  world  \
doc_title chunk_id                 ...                                    
ASHPUTTEL 0              0      0  ...      0     0      0     0      0   
          1              0      0  ...      0     0      0     1      0   
          2              0      0  ...      0     0      0     0      0   
          3              0      0  ...      0     0      0     0      0   
          4              0      0  ...      0     0      0     0      0   
...                    ...    ...  ...    ...   ...    ...   ...    ...   
TOM THUMB 15             0      0  ...      0     0      0     0      0   
          16             0      0  ...      0     0      0     0      0   
          17             0      0  ...      0     0      0     0      0   
          18             0      0  ...      0     0      0     0      2   
          19             0      0  ...      0     0      0     0      0   

                    wretch  yard  year  years  youth  
doc_title chunk_id                                    
ASHPUTTEL 0              0     0     0      0      0  
          1              0     0     0      0      0  
          2              0     0     0      0      0  
          3              0     0     0      0      0  
          4              0     0     0      0      0  
...                    ...   ...   ...    ...    ...  
TOM THUMB 15             0     0     0      0      0  
          16             0     0     0      0      0  
          17             0     0     0      0      0  
          18             0     0     0      0      0  
          19             0     0     0      0      0  

[784 rows x 707 columns]

In [252]:
# Used claude code to help with tfidf engine and model because I want nmf to get better topics than lda
# tfidf_engine = TfidfVectorizer(max_df=.75, min_df=5, stop_words=my_stop_words)
# tfidf_model = tfidf_engine.fit_transform(DOCS.doc_str)
# TERMS = tfidf_engine.get_feature_names_out()
# TFIDF = tfidf_engine.fit_transform(DOCS.doc_str)
# VOCAB = pd.DataFrame(index=TERMS)
# VOCAB.index.name = 'term_str'
# DTM = pd.DataFrame(tfidf_model.toarray(), index=DOCS.index, columns=TERMS)
# DTM


## Generate Model with 20 Topics

In [253]:
n_topics = 5
max_iter = 100
n_top_terms = 5
TNAMES = [f"T{str(x).zfill(len(str(n_topics)))}" for x in range(n_topics)]

In [254]:
if model_type == 'lda':
    topic_engine = LDA(n_components=n_topics, max_iter=max_iter)
elif model_type == 'nmf':
    topic_engine = NMF(n_components=n_topics, max_iter=max_iter)
topic_model = topic_engine.fit_transform(count_model)

In [255]:
model_type

'lda'

## THETA

In [256]:
THETA = pd.DataFrame(topic_model, index=DOCS.index, columns=TNAMES)
THETA.columns.name = 'topic_id'
THETA.sample(10).T.style.background_gradient(cmap=colors, axis=None)

doc_title,THE BLUE LIGHT,THE RAVEN,TOM THUMB,BRIAR ROSE,THE KING OF THE GOLDEN MOUNTAIN,CAT-SKIN,ASHPUTTEL,SNOW-WHITE AND ROSE-RED,HANSEL AND GRETEL,THE WATER OF LIFE
chunk_id,8,9,1,9,13,16,9,1,14,13
topic_id,,,,,,,,,,
T0,0.008822,0.936008,0.590000,0.008140,0.269867,0.020263,0.148579,0.333159,0.008508,0.594804
T1,0.009032,0.015731,0.161050,0.508820,0.010929,0.309503,0.009295,0.011277,0.008580,0.012018
T2,0.964111,0.015681,0.017273,0.008174,0.010704,0.020245,0.009237,0.011221,0.008565,0.011966
T3,0.009202,0.016509,0.017359,0.097076,0.697671,0.629726,0.823620,0.633049,0.008522,0.369097
T4,0.008832,0.016071,0.214319,0.377789,0.010828,0.020262,0.009268,0.011294,0.965824,0.012115


In [257]:
THETA_vol=THETA.join(LIB)

THETA_vol_agg=THETA_vol.groupby('volume').mean()
top_volume=THETA_vol_agg.idxmax()
# topic_ideas = {
#     'T0':'T0: matriarchy/domesticity',
#     'T1': 'T1: patriarchy/family',
#     'T2': 'T2: outdoors',
#     'T3': 'T3:family/time',
#     'T4': 'T4: princess tales'
# }
# THETA_vol.rename(columns=topic_ideas, inplace=True)
THETA_vol_agg.style.background_gradient(cmap=colors, axis=None)


,T0,T1,T2,T3,T4
volume,,,,,
1,0.214746,0.197355,0.187326,0.210096,0.190476
2,0.269266,0.161727,0.194901,0.200078,0.174029


## PHI

In [258]:
PHI = pd.DataFrame(topic_engine.components_, columns=TERMS, index=TNAMES)
PHI.index.name = 'topic_id'
PHI.columns.name = 'term_str'
PHI.T.sample(10).T.style.background_gradient(cmap=colors, axis=None)

term_str,marleen,coffin,pillow,soldiers,son,girl,fall,vain,rock,cellar
topic_id,,,,,,,,,,
T0,2.648089,0.201625,0.200823,1.307767,27.059071,11.012966,5.174902,4.401405,0.202312,0.200547
T1,0.204188,8.186557,0.200371,3.275431,17.147656,13.016462,1.196530,0.201963,0.200004,15.815937
T2,0.200276,0.202920,7.200373,0.200008,0.201314,29.926841,2.828664,7.726069,0.200161,6.583150
T3,3.747443,0.200005,0.200814,4.235705,85.390730,7.840558,3.593671,0.203281,1.758039,0.200024
T4,0.200004,0.208893,2.197619,0.981089,0.201229,0.203173,0.206232,8.467282,8.639484,0.200342


## Get Top Terms By Topic

In [ ]:
TOPICS = PHI.stack().groupby('topic_id')\
    .apply(lambda x: ' '.join(x.sort_values(ascending=False).head(n_top_terms).reset_index().term_str))\
    .to_frame('top_terms')
TOPICS


NameError: name 'seed' is not defined

## PCA + LDA

In [ ]:
THETA_vol

T0        T1        T2        T3        T4  volume
doc_title chunk_id                                                          
ASHPUTTEL 0         0.305052  0.008476  0.008588  0.283725  0.394160       1
          1         0.009398  0.009252  0.009216  0.962807  0.009327       1
          2         0.967255  0.008127  0.008103  0.008242  0.008274       1
          3         0.245894  0.011249  0.153755  0.011269  0.577833       1
          4         0.325502  0.016821  0.016714  0.375178  0.265786       1
...                      ...       ...       ...       ...       ...     ...
TOM THUMB 15        0.011334  0.519635  0.446503  0.011204  0.011324       1
          16        0.013559  0.356057  0.603409  0.013494  0.013481       1
          17        0.009738  0.428655  0.542002  0.009921  0.009684       1
          18        0.012739  0.404628  0.012890  0.012802  0.556941       1
          19        0.017125  0.016909  0.017003  0.016824  0.932138       1

[784 rows x 6 columns]

In [ ]:
THETA.join(LIB)

T0        T1        T2        T3        T4  volume
doc_title chunk_id                                                          
ASHPUTTEL 0         0.305052  0.008476  0.008588  0.283725  0.394160       1
          1         0.009398  0.009252  0.009216  0.962807  0.009327       1
          2         0.967255  0.008127  0.008103  0.008242  0.008274       1
          3         0.245894  0.011249  0.153755  0.011269  0.577833       1
          4         0.325502  0.016821  0.016714  0.375178  0.265786       1
...                      ...       ...       ...       ...       ...     ...
TOM THUMB 15        0.011334  0.519635  0.446503  0.011204  0.011324       1
          16        0.013559  0.356057  0.603409  0.013494  0.013481       1
          17        0.009738  0.428655  0.542002  0.009921  0.009684       1
          18        0.012739  0.404628  0.012890  0.012802  0.556941       1
          19        0.017125  0.016909  0.017003  0.016824  0.932138       1

[784 rows x 6 columns]

In [ ]:
pca_engine = PCA(n_components=4)
TCM = pd.DataFrame(pca_engine.fit_transform(THETA.T),index=THETA.T.index)
TCM.columns = ['PC{}'.format(i) for i in TCM.columns]
TCM['doc_mean_weight']= TCM.mean(axis=1)
TCM['top_volume'] = top_volume
TCM.style.background_gradient(cmap=colors, axis=None)


,PC0,PC1,PC2,PC3,doc_mean_weight,top_volume
topic_id,,,,,,
T0,-2.032739,-1.032665,6.721138,-3.712600,-0.014216,1
T1,-3.677364,7.454243,-2.641122,-0.848656,0.071776,1
T2,-2.773068,-5.830966,-5.025625,-2.071394,-3.925263,1
T3,-1.590541,-1.293938,1.672823,7.135226,1.480892,2
T4,10.073711,0.703326,-0.727215,-0.502576,2.386812,2


In [ ]:
def vis_pcs(a, b,DCM,):
    fig =px.scatter(DCM, 
        f"PC{a}", f"PC{b}", 
        color=DCM['top_volume'].astype('category'), 
        size= np.abs(DCM['doc_mean_weight']),
        # hover_name=DCM.title_para,
        marginal_x='box', 
        height=1000, 
        width=1200)
    return fig


In [ ]:
vis_pcs(1,2,TCM)

## Save Files to Output

In [ ]:
THETA.to_csv(f"{output_dir}/pg2591-THETA.csv", index=True)
PHI.to_csv(f"{output_dir}/pg2591-PHI.csv", index=True)
DTM_chunk.to_csv(f"{output_dir}/pg2591-CHUNKED_DTM.csv", index=True)
TOPICS.to_csv(f"{output_dir}/pg2591-TOPICS.csv", index=True)